03 .Ordination 
==============  
**Overview of relationships between samples and features** 

In [ ]:
library(phyloseq)
suppressPackageStartupMessages(library(plyr))
suppressPackageStartupMessages(library(vegan))
suppressPackageStartupMessages(library(DESeq2))
suppressPackageStartupMessages(library(ggplot2))

**Low dimensional representation of relationships between samples/features**  
We are going to follow closely phyloseq's ordination examples (https://joey711.github.io/phyloseq/plot_ordination-examples.html)  
Read data:  


In [ ]:
#rds <- readRDS("../../../../../data/tmp/microbiome/quant/qiime2/silva/physeq_filtered.rds")


In [ ]:
rds <- readRDS(snakemake@input[[1]])

In [ ]:
df <- data.frame(as(sample_data(rds), "matrix"))
group <- factor(df[,snakemake@params$condition])
cat("Condition: ", snakemake@params$condition, "\n")
cat("Summary:\n")
print(summary(group))
if (!is.null(snakemake@params$ref_level)){
    cat("Setting", snakemake@params$ref_level, "as reference level", "\n")
    group <- relevel(group, ref=snakemake@params$ref_level)
}
df[,"Sample_Group"] <- group
sample_data(rds) <- df


In [ ]:
df <- data.frame(as(sample_data(rds), "matrix"))
batch <- NULL
if (!is.null(snakemake@params$batch)){
    print(cat("\nBatch factor given as: ", snakemake@params$batch, "\n"))
    batch <- factor(snakemake@params$batch)
    print(summary(batch))
    df[,snakemake@params$batch] <- batch
    sample_data(rds) <- df
}


---------------

Next step is to agglomerate at a specific taxon level 

In [ ]:
if (!is.null(snakemake@params$taxrank)){
    cat("Taxrank: ", snakemake@params$taxrank, "\n")
    if (require(speedyseq)){
        rds <- speedyseq::tax_glom(rds, taxrank=snakemake@params$taxrank)
    } else{
        rds <- tax_glom(rds, taxrank=snakemake@params$taxrank)
    } 
} else{
    cat("Taxrank: None, Data is not aggregated. Each feature is an ASV (specific sequence)", "\n")
}
rds

---------------

It is normal to filter a bit more aggresive in dimension reduction methods (as compared to diff. epxrs for instance) as we are looking for variance across all groups to identify global patterns. Also, we may work on a subset and thus prevalence thresholds may have changed.

In [ ]:

if (!is.null(snakemake@params$independent_filtering)){
    if (snakemake@params$independent_filtering == TRUE){
        wh0 = genefilter_sample(rds, filterfun_sample(function(x) x > 5), A=0.25*nsamples(rds))
        rds = prune_taxa(wh0, rds)
        phylum.sum = tapply(taxa_sums(rds), tax_table(rds)[, "Phylum"], sum, na.rm=TRUE)
        top6phyla = names(sort(phylum.sum, TRUE))[1:6]
        rds = prune_taxa((tax_table(rds)[, "Phylum"] %in% top6phyla), rds)
    }
}

cat("After filtering:\n")
rds

We like to start out with equal impact of all samples and start out with a normalization to equal count (Counts per million)

In [ ]:
rds = transform_sample_counts(rds, function(x) 1E6 * x/sum(x))

In [ ]:
#https://github.com/LieberInstitute/jaffelab
cleaningY <- function(y, mod, P) {
    stopifnot(P <= ncol(mod))
    Hat <- solve(t(mod) %*% mod) %*% t(mod)
    ## For dealing with NAs
    ## https://stackoverflow.com/questions/16535084/matrix-multiplication-with-scattered-na-values
    ty <- t(y)
    ty[is.na(ty)] <- 0
    beta <- (Hat %*% ty)
    ## Note that y might still have the NAs, and NA - a number = NA
    ## so there's no need to reset the NAs back on cleany
    cleany <- y - t(as.matrix(mod[, -c(seq_len(P))]) %*% beta[-seq_len(P), ])
    return(cleany)
    }

if (!is.null(snakemake@params$block)){
    df <- data.frame(as(sample_data(rds), "matrix"))
    mod <- with(df, model.matrix(~ snakemake@params$condition + snakemake@params$batch))
    Y <- t(as(otu_table(rds), "matrix"))
    Yclean <- cleaningY(Y, mod, P=ncol(mod))
    otu_table(rds) <- Yclean
    }


--------------------


Our default ordination methods is using Bray-Curtis distance (phylogenetic distance) and Non-negative multidimensional scaling (NMDS) as the dimension reduction method.

In [ ]:
GP.ord <- ordinate(rds, "NMDS", "bray")

We start out by looking at the relations between taxa to look for any irregular patterns. Each point is either an OTU/ASV or summarized to a specific taxonomic level given by model name. 

In [ ]:
p1 = plot_ordination(rds, GP.ord, type="taxa", color="Phylum", title="taxa")
print(p1)

In [ ]:
p1 + facet_wrap(~Phylum, 3)

Do the same figures for samples. Each point is one sample.

In [ ]:
p2 = plot_ordination(rds, GP.ord, type="samples", color="Sample_Group", shape="Sample_Group") 
p2 + geom_point(size=7) + ggtitle("samples")

In [ ]:
p4 = plot_ordination(rds, GP.ord, type="split", color="Phylum", shape="Sample_Group", label="Sample_Group", title="biplot") 
p4

Lets do a summary of all the available decompositions in phyloseq:

In [ ]:
dist = "bray"
ord_meths = c("DCA", "CCA", "RDA", "DPCoA", "NMDS", "MDS", "PCoA")
plist = llply(as.list(ord_meths), function(i, physeq, dist){
        ordi = ordinate(rds, method=i, distance=dist)
        plot_ordination(rds, ordi, "samples", color="Sample_Group")
}, rds, dist)
names(plist) <- ord_meths

pdataframe = ldply(plist, function(x){
    df = x$data[, 1:2]
    colnames(df) = c("Axis_1", "Axis_2")
    return(cbind(df, x$data))
})
names(pdataframe)[1] = "method"

In [ ]:
p = ggplot(pdataframe, aes(Axis_1, Axis_2, color=Sample_Group, shape=Sample_Group, fill=Sample_Group))
p = p + geom_point(size=4)
p = p + facet_wrap(~method, scales="free")
p = p + scale_fill_brewer(type="qual", palette="Set1")
p = p + scale_colour_brewer(type="qual", palette="Set1")
p

An example with weighted Unifrac (can be a bit more unstable than Bray) using Principal Coordinate Analysis

In [ ]:
ordu = ordinate(rds, "PCoA", "unifrac", weighted=TRUE)
p5 <- plot_ordination(rds, ordu, color="Sample_Group", shape="Sample_Group")
p5 = p5 + geom_point(size=7,alpha=0.8)
p5 = p5 + scale_colour_brewer(type="qual", palette="Set1")
p5 + ggtitle("MDS/PCoA on weighted-UniFrac distance")

----------------------

Lets check for significance of our *condition* in the low dimensional space defined bu the ordination. Are the groups signficant different from eachother? For this we are using permuation ANOVA from the *vegan* package (adonis). We will extract the two first dimensions (PC) and use this for signficance testing.

In [ ]:
PC <- ordu$vectors[,1:2]
PC <- GP.ord$points
X <- cbind(PC, df)
adonis(PC ~ Sample_Group, data = X, method='eu')

Write the default dimred coordinates to file

In [ ]:
write.table(X, file=snakemake@output[[1]])

In [ ]:
cat("Wrote:", snakemake@output[[1]])